In [1]:
# Requiero Mesa > 3.3
# Importamos las clases que se requieren para manejar los agentes (Agent) y su entorno (Model).
# Cada modelo puede contener múltiples agentes.
from mesa import Agent, Model

# Debido a que necesitamos que existe un solo agente por celda, elegimos ''SingleGrid''.
from mesa.space import SingleGrid

# Haremos uso de ''DataCollector'' para obtener información de cada paso de la simulación.
from mesa.datacollection import DataCollector

# Haremos uso de ''batch_run'' para ejecutar varias simulaciones
from mesa.batchrunner import batch_run

# matplotlib lo usaremos crear una animación de cada uno de los pasos del modelo.
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
plt.rcParams["animation.html"] = "jshtml"
matplotlib.rcParams['animation.embed_limit'] = 2**128

# Importamos los siguientes paquetes para el mejor manejo de valores numéricos.
import numpy as np
import pandas as pd

c:\Users\inaki\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class ExploradorAgent(Agent):
    def __init__(self, model):
        super().__init__(model)
        self.energia = 0
        self.llevandoRecurso = 0

    def move(self):
        # Verificar si hay energia disponible
        if self.energia > 0:
            
            # Hay que agregar el verificar si lleva recurso
            # si lleva recurso dirigirse a la base
            if not self.llevandoRecurso == 1:

                # Encontrar las posiciones posibles y hacer una permutacion para posteriormente elegir una al azar
                possible_positions = self.model.grid.get_neighborhood(self.pos, moore=False, include_center=False)
                options = np.random.permutation(len(possible_positions))

                # Si en las opciones hay una celda libre moverse hacia esa celda
                for i in options:
                    position = possible_positions[i]
                    (x, y) = position
                    if self.model.grid.is_cell_empty(position) == True:
                        self.model.grid.move_agent(self, position)
                        break
                # Quitar uno de energia por movimiento
                self.energia -= 1

            else:
                current_x, current_y = self.pos
                base_x, base_y = self.model.base

                # Calcular diferencia entre coordenadas del agente y de la base
                dx = base_x - current_x
                dy = base_y - current_y

                # Priorizar X, cambiar si el camino esta bloqueado
                if dx != 0:
                    next_x = current_x + (1 if dx > 0 else -1)
                    next_pos = (next_x, current_y)
                elif dy != 0:
                    next_y = current_y + (1 if dy > 0 else -1)
                    next_pos = (current_x, next_y)
                
                # Ya esta en la base
                else:
                    next_pos = self.pos

                if self.model.grid.is_cell_empty(next_pos):
                    self.model.grid.move_agent(self, next_pos)
                    self.energia -= 1


    def step(self):
        self.move()

In [ ]:
class ExploradorModel(Model):
    def __init__(self, width=11, height=11, agents=5, resources=20):
        super().__init__()

        self.grid = SingleGrid(width, height, torus=False)
        # Crear dataCollector

        # Establecer posicion de la base
        self.base = (width // 2, height // 2)

        # Crea matriz de ceros y agrega recursos aleatoriamente
        self.cells = np.zeros( (width, height))
        count = int( ( width * height * ( resources / 100 ) ) )
        while (count > 0):
            x = self.random.randrange(width)
            y = self.random.randrange(height)
            if self.cells[x][y] == 0 and self.cells[x][y] != self.base:
                self.cells[x][y] = 1
                count -= 1  


        